In [1]:
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook

# ============================================================
# CONFIGURAZIONE
# ============================================================

CARTELLA = r"C:\Users\KK00919\OneDrive - Cherry Bank\Desktop\KPI\database\fonti"
CARTELLA_OUTPUT = r"C:\Users\KK00919\OneDrive - Cherry Bank\Desktop\KPI\database"
NOME_FILE = "database_v1_filtrato.xlsx"
OUTPUT = Path(CARTELLA_OUTPUT) / NOME_FILE

FILE_FOGLI = {
    "Dim_task.xlsx": [("Dimensionamento", "Dimensionamento"), ("Task", "Task")],
    "bancassurance.xlsx": [("Export", "10001100023100042100129")],
    "base_dati_adv_2026.xlsx": [("Sheet0", "10001100023100034"), ("Sheet0", "10004100067100081v2")],
    "bonifici_banca.xlsx": [("Export", "10001100023100036v1")],
    "bonifici_estero.xlsx": [("Foglio1", "10001100023100036v4")],
    "bonifici_raisin.xlsx": [("Export", "10001100023100042100130v3")],
    "cambiali_circolari_tesoreria.xlsx": [("Export", "10001100023100036v2")],
    "cassette.xlsx": [("Export", "10001100023100036v3")],
    "Censimenti 2026.xlsx": [("Export0706", "10001100023000001"), ("Export0706", "10004100067100081v1")],
    "fct_monitoraggio.xlsx": [("Export", "10004100067100080v1")],
    "debitori.xlsx": [("Export", "10004100067100080v2")],
    "n_frodi.xlsx": [("Export", "10001100023100044"), ("Export", "10001100023100042100130v2")],
    "pef.xlsx": [("database", "10004100052"), ("database", "10004100051"), ("database", "10004100067100078v1")],
    "perfezionamenti.xlsx": [("Export", "10004100067100079"), ("Export", "10004100067100081v4")],
    "rapporti_digital.xlsx": [("Export", "10001100023100042100130v1")],
    "rapporti_cib.xlsx": [("Export", "10004100067100081v3")],
    "collegamenti.xlsx": [("Export", "10004100067100078v2")]
}

# Codice foglio finale -> nome logico usato nelle configurazioni.
# L'ordine determina l'ordine dei fogli nell'Excel.
SHEET_NAME_MAP = {
    "10001100023100042100130v1": "digital_rapporti",
    "10001100023100042100130v2": "digital_frodi",
    "10001100023100042100130v3": "digital_raisin",
    "10001100023100042100129": "bancassurance",
    "10001100023100036v1": "monetica_bonifici_banca",
    "10001100023100036v2": "monetica_cassa",
    "10001100023100036v3": "monetica_cassette",
    "10001100023100036v4": "monetica_bonifici_estero",
    "10001100023000001": "anagrafe",
    "10001100023100034": "ops_aml",
    "10001100023100044": "antifrode",
    "10004100067100078v1": "fidi",
    "10004100067100078v2": "fidi_collegamenti",
    "10004100067100079": "perfezionamenti",
    "10004100067100080v1": "factoring_cedenti",
    "10004100067100080v2": "factoring_debitori",
    "10004100067100081v1": "specialty_censimenti",
    "10004100067100081v2": "specialty_adv",
    "10004100067100081v3": "specialty_rapporti",
    "10004100067100081v4": "specialty_perfezionamenti",
    "10004100052": "credito",
    "10004100051": "credito_speciale",
}

# ============================================================
# FILTRI RIGHE
# ============================================================

ROW_FILTERS = {
    "digital_frodi": lambda df: df["Campo personalizzato (U.O. segnalante)"] == "DIGITAL BANK",
    "ops_aml": lambda df: df["fascia_rischio"] == "Alta",
    "factoring_debitori": lambda df: (
        (df["descrizione_stato_linea"].astype(str).str.strip() == "Deliberata operativa") &
        ((pd.to_numeric(df["accordato_pro_solvendo"], errors="coerce").fillna(0) != 0) |
         (pd.to_numeric(df["accordato_pro_soluto"], errors="coerce").fillna(0) != 0))
    ),
    "specialty_adv": lambda df: df["business_unit"].astype(str).str.strip() == "Corporate Banking",
    "specialty_perfezionamenti": lambda df: df["des_business_unit"].astype(str).str.strip().isin([
        "Finanza Strutturata",
        "Special Situations",
        "Turnaround & Strategic Finance",
    ]),
}

# ============================================================
# TRASFORMAZIONI
# ============================================================

TRANSFORMATIONS = {
    "antifrode": lambda df: (
        df.assign(mese=pd.to_datetime(df["Campo personalizzato (Data operazione (FR))"], errors="coerce").dt.to_period("M").astype(str))
        .groupby(["mese", "Campo personalizzato (Cluster Frode Banca)", "Campo personalizzato (Classificazione pratica)"], dropna=False)["Campo personalizzato (NDG (FR))"]
        .count()
        .reset_index(name="conteggio")
        .rename(columns={
            "Campo personalizzato (Cluster Frode Banca)": "cluster_frode",
            "Campo personalizzato (Classificazione pratica)": "classificazione"
        })
    ),
}

# ============================================================
# FILTRI COLONNE
# ============================================================

COLUMNS_TO_KEEP = {
    "credito": ["des_tipo_istruttoria", "des_stato_istruttoria", "nro_giorni_lavorazione", "nro_giorni_coda", "des_ndg_operatore_lavorazione", "des_organo_delib", "des_scopo_pratica", "des_tipo_delibera", "dta_delibera", "dta_istruttoria", "des_ruolo_operatore_apertura"],
    "credito_speciale": ["des_tipo_istruttoria", "des_stato_istruttoria", "nro_giorni_lavorazione", "nro_giorni_coda", "des_ndg_operatore_lavorazione", "des_organo_delib", "des_scopo_pratica", "des_tipo_delibera", "dta_delibera", "dta_istruttoria"],
    "perfezionamenti": ["des_business_unit", "dta_delibera", "dta_operativa", "cod_identif_fido", "des_forma_tecnica"],
    "factoring_cedenti": ["ndg", "business unit", "filiale", "data prima stipula", "accordato", "impiego", "turnover anno corrente"],
    "factoring_debitori": ["ndg_debitore", "data_delibera", "accordato_pro_solvendo", "accordato_pro_soluto", "descrizione_prodotto"],
    "anagrafe": ["des_business_unit", "dta_censimento", "des_natura_giuridica", "des_status_generic"],
    "antifrode": ["mese", "cluster_frode", "classificazione", "conteggio"],
    "ops_aml": ["fascia_rischio", "data_uscita", "data_inserimento", "data_scadenza_adv", "ndg", "business_unit", "cluster", "workflow", "tipo_verifica"],
    "bancassurance": ["data_ordine", "descrizione_stato", "tot_generale_euro"],
    "digital_rapporti": ["dta_rapporto_apert", "des_categoria_rapporto", "dta_rapporto_estinzione"],
    "digital_frodi": ["Campo personalizzato (Data operazione (FR))", "Campo personalizzato (Cluster Frode Banca)"],
    "digital_raisin": ["data_inserimento"],
    "monetica_bonifici_banca": ["data_valuta_fissa_al_beneficiario", "data_regolamento", "importo_bonifico"],
    "monetica_cassa": ["d_data_cont", "tg04_causale1", "descrizione_filiale"],
    "monetica_cassette": ["dta_rapporto_apert", "des_business_unit"],
    "monetica_bonifici_estero": ["data_inserimento", "importo_bonifico", "paese_ord", "paese_beneficiario"],
    "fidi": ["dta_istruttoria"], # conto tutte le pef istruite se le riclassificazioni le fanno in ogni caso e per ogni BU
    "fidi_collegamenti": ["cod_ndg_rete", "dta_censim_collegato"],
    "specialty_censimenti": ["des_business_unit", "dta_censimento"],
    "specialty_adv": ["fascia_rischio", "data_uscita", "ndg", "business_unit"],
    "specialty_rapporti": ["dta_rapporto_apert", "des_business_unit"],
    "specialty_perfezionamenti": ["des_business_unit", "dta_operativa", "cod_identif_fido"],
}

# ============================================================
# FUNZIONI
# ============================================================

def read_sheet(percorso, foglio):
    if foglio == "Dimensionamento":
        return pd.read_excel(percorso, sheet_name=foglio, dtype={0: str}, keep_default_na=False)
    return pd.read_excel(percorso, sheet_name=foglio)

def apply_row_filter(df, sheet_key):
    if sheet_key not in ROW_FILTERS:
        return df, 0
    iniziali = len(df)
    try:
        df = df[ROW_FILTERS[sheet_key](df)].copy()
        return df, iniziali - len(df)
    except Exception as e:
        print(f"   ❌ Errore filtro righe '{sheet_key}': {e}")
        return df, 0

def apply_transformation(df, sheet_key):
    if sheet_key not in TRANSFORMATIONS:
        return df
    try:
        return TRANSFORMATIONS[sheet_key](df).copy()
    except Exception as e:
        print(f"   ❌ Errore trasformazione '{sheet_key}': {e}")
        return df

def select_columns(df, sheet_key):
    if sheet_key not in COLUMNS_TO_KEEP:
        return df, [], []
    richieste = COLUMNS_TO_KEEP[sheet_key]
    presenti = [col for col in richieste if col in df.columns]
    mancanti = [col for col in richieste if col not in df.columns]
    return df[presenti].copy(), presenti, mancanti

# ============================================================
# ELABORAZIONE
# ============================================================

def main():
    print("=" * 80)
    print("🚀 INIZIO CREAZIONE DATABASE")
    print("=" * 80)

    Path(CARTELLA_OUTPUT).mkdir(parents=True, exist_ok=True)
    fogli = {}

    # LETTURA
    for nome_file, configurazioni in FILE_FOGLI.items():
        percorso = Path(CARTELLA) / nome_file
        if not percorso.exists():
            print(f"⚠️ File non trovato: {nome_file}")
            continue

        for foglio_origine, codice in configurazioni:
            try:
                df = read_sheet(percorso, foglio_origine)
                fogli[codice] = {"df": df, "file": nome_file, "foglio": foglio_origine}
                print(f"✅ Letto: {nome_file} → {foglio_origine} → {codice}")
            except Exception as e:
                print(f"❌ Errore: {nome_file} / {foglio_origine}: {e}")

    # SCRITTURA
    with pd.ExcelWriter(OUTPUT, engine="openpyxl") as writer:

        # Dimensionamento
        if "Dimensionamento" in fogli:
            df = fogli["Dimensionamento"]["df"].copy()
            df.to_excel(writer, sheet_name="Dimensionamento", index=False)
            print(f"📄 [1] Dimensionamento → {len(df):,} righe × {len(df.columns)} colonne")
        else:
            print("⚠️ Dimensionamento non trovato")

        # Task
        if "Task" in fogli:
            df = fogli["Task"]["df"].copy()
            df.to_excel(writer, sheet_name="Task", index=False)
            print(f"📄 [2] Task → {len(df):,} righe × {len(df.columns)} colonne")
        else:
            print("⚠️ Task non trovato")

        # Altri fogli: ROW FILTER → TRANSFORMATION → DROP COLONNE
        posizione = 3

        for codice, nome_logico in SHEET_NAME_MAP.items():
            if codice not in fogli:
                print(f"⚠️ {codice} ({nome_logico}): sorgente non trovata")
                continue

            info = fogli[codice]
            df = info["df"].copy()

            # 1. FILTRO RIGHE
            df, righe_eliminate = apply_row_filter(df, nome_logico)

            # 2. TRASFORMAZIONE
            df = apply_transformation(df, nome_logico)

            # 3. DROP / SELEZIONE COLONNE
            df, colonne_presenti, colonne_mancanti = select_columns(df, nome_logico)

            if not colonne_presenti:
                print(f"❌ {codice} ({nome_logico}): nessuna colonna trovata")
                continue

            # 4. SCRITTURA
            df.to_excel(writer, sheet_name=codice, index=False)

            print(f"📄 [{posizione}] {codice} ({nome_logico}) → {len(df):,} righe × {len(colonne_presenti)} colonne")
            if righe_eliminate:
                print(f"   ├─ 🗑️ Righe eliminate: {righe_eliminate:,}")
            if colonne_mancanti:
                print(f"   ├─ ⚠️ Colonne mancanti: {colonne_mancanti}")
            print(f"   └─ Origine: {info['file']} → {info['foglio']}")

            posizione += 1

    # DIMENSIONAMENTO: prima colonna come testo
    try:
        wb = load_workbook(OUTPUT)
        if "Dimensionamento" in wb.sheetnames:
            for cell in wb["Dimensionamento"]["A"]:
                cell.number_format = "@"
        wb.save(OUTPUT)
        wb.close()
        print("✅ Prima colonna di Dimensionamento impostata come testo")
    except Exception as e:
        print(f"⚠️ Errore impostazione testo Dimensionamento: {e}")

    # RIEPILOGO
    print("\n" + "=" * 80)
    print("✅ ELABORAZIONE COMPLETATA")
    print("=" * 80)
    print(f"📄 File: {OUTPUT}")

    try:
        wb = load_workbook(OUTPUT, read_only=True)
        print("\n📑 Ordine fogli:")
        for i, nome in enumerate(wb.sheetnames, 1):
            print(f"   {i}. {nome}")
        wb.close()
    except Exception as e:
        print(f"⚠️ Errore lettura riepilogo: {e}")


if __name__ == "__main__":
    main()

🚀 INIZIO CREAZIONE DATABASE
✅ Letto: Dim_task.xlsx → Dimensionamento → Dimensionamento
✅ Letto: Dim_task.xlsx → Task → Task
✅ Letto: bancassurance.xlsx → Export → 10001100023100042100129
✅ Letto: base_dati_adv_2026.xlsx → Sheet0 → 10001100023100034
✅ Letto: base_dati_adv_2026.xlsx → Sheet0 → 10004100067100081v2
✅ Letto: bonifici_banca.xlsx → Export → 10001100023100036v1
✅ Letto: bonifici_estero.xlsx → Foglio1 → 10001100023100036v4
✅ Letto: bonifici_raisin.xlsx → Export → 10001100023100042100130v3
✅ Letto: cambiali_circolari_tesoreria.xlsx → Export → 10001100023100036v2
✅ Letto: cassette.xlsx → Export → 10001100023100036v3
✅ Letto: Censimenti 2026.xlsx → Export0706 → 10001100023000001
✅ Letto: Censimenti 2026.xlsx → Export0706 → 10004100067100081v1
✅ Letto: fct_monitoraggio.xlsx → Export → 10004100067100080v1
✅ Letto: debitori.xlsx → Export → 10004100067100080v2
✅ Letto: n_frodi.xlsx → Export → 10001100023100044
✅ Letto: n_frodi.xlsx → Export → 10001100023100042100130v2
✅ Letto: pef.xls